
# Hyperparam_Optimierung — 5 Modelle × 3 Komplexitäten (15 Läufe)

**Neu & Anforderungen**

- Optimierung für **alle `COMPLEXITY_PRESETS`** aus `experiment_pipeline_multiconfig.py` (z. B. simple/medium/high).
- **Keine** Optimierung der Komplexitäts-Parameter (Architektur/Kapazität) und **keine** Feature-Optimierung.
- **Optimiert werden**: `lags` **und** modell-spezifische **Trainings-/Reg-Parameter**, die **sinnvoll** sind.
- **Feste Parameter**:  
  - `train_fraction = 0.8`  
  - `base_features = ["Group4-2_S6_VolumetricFlowRate", "Group4-2_S6_MassFlowRate"]`  
  - `time_features = []`  
  - `target_feature = "Group4-2_S6_VolumetricFlowRate"`  
  - `include_roll_mean = True`, `include_roll_std = True`, **Fenster** via `rolling_window_size = 2`  
- Während der Optimierung: `inference_interval_sec = 0`.
- **Keine Persistenz** von Modellen oder Fehlermetriken – es werden **nur die besten Hyperparameter** gespeichert.
- **Dataset-Pfad** wird **wie in der Pipeline** über `CONFIG_PATH['paths']` ermittelt.


In [1]:

# ✅ Bootstrap: Optuna automatisch installieren, falls nicht vorhanden
try:
    import optuna  # noqa: F401
except Exception:
    import sys, subprocess
    print("Optuna nicht gefunden — Installation wird versucht (pip install optuna)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "--quiet"])
    import optuna  # noqa: F401
print("Optuna ist verfügbar.")


Optuna ist verfügbar.


c:\DEV\RevPi_ML\ML_Edge_Device\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

import os, sys, json, importlib, warnings, re
from pathlib import Path
import numpy as np
import pandas as pd
import optuna

ROOT = Path.cwd()
sys.path.append(str(ROOT))
sys.path.append('/mnt/data')

# Robust import of experiment config + presets
try:
    epm = importlib.import_module('experiment_pipeline_multiconfig')
except ModuleNotFoundError:
    from pathlib import Path
    import importlib.util
    print("experiment_pipeline_multiconfig nicht im sys.path gefunden — versuche alternative Pfade...")
    candidate_dirs = [
        Path.cwd(), Path.cwd() / "experiment",
        Path.cwd().parent, Path.cwd().parent / "experiment",
        Path(r"C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\experiment"),
        Path('/mnt/data'),
    ]
    cur = Path.cwd()
    for _ in range(5):
        candidate_dirs += [cur, cur / "experiment", cur / "experiments"]
        cur = cur.parent
    seen = set(); dirs = []
    for d in candidate_dirs:
        try:
            dr = d.resolve()
        except Exception:
            continue
        if dr.exists() and dr not in seen:
            seen.add(dr); dirs.append(dr)
    epm = None
    for d in dirs:
        if (d / "experiment_pipeline_multiconfig.py").exists():
            if str(d) not in sys.path:
                sys.path.insert(0, str(d))
            try:
                epm = importlib.import_module('experiment_pipeline_multiconfig')
                print(f"✓ importiert aus: {d}")
                break
            except Exception as e:
                print(f"Fehlgeschlagen in {d}: {e}")
    if epm is None:
        for d in dirs:
            fp = d / "experiment_pipeline_multiconfig.py"
            if fp.exists():
                spec = importlib.util.spec_from_file_location("experiment_pipeline_multiconfig", fp)
                mod = importlib.util.module_from_spec(spec)
                try:
                    spec.loader.exec_module(mod)  # type: ignore
                    epm = mod
                    print(f"✓ via direktem Pfad geladen: {fp}")
                    break
                except Exception as e:
                    print(f"Direktlade-Fehler bei {fp}: {e}")
    if epm is None:
        raise ModuleNotFoundError("Konnte 'experiment_pipeline_multiconfig.py' nicht finden.")
COMPLEXITY_PRESETS = getattr(epm, 'COMPLEXITY_PRESETS')
BASE_COMMON = getattr(epm, 'BASE_COMMON')
TRAINER_MAP = getattr(epm, 'TRAINER_MAP')
algorithm_to_folder = getattr(epm, 'algorithm_to_folder')
build_training_config = getattr(epm, 'build_training_config')


experiment_pipeline_multiconfig nicht im sys.path gefunden — versuche alternative Pfade...
✓ importiert aus: C:\DEV\RevPi_ML\ML_Edge_Device\experiment


In [3]:

# Globale Projektpfade (wie in der Pipeline)
try:
    from config.config_general import CONFIG_PATH
except ModuleNotFoundError:
    from config_general import CONFIG_PATH


In [4]:

FIXED_FEATURES = {
    "train_fraction": 0.7,
    "base_features": ["Group4-2_S6_VolumetricFlowRate", "Group4-2_S6_MassFlowRate"],
    "time_features": [],
    "target_feature": "Group4-2_S6_VolumetricFlowRate",
    "include_roll_mean": True,
    "include_roll_std": True,
    "rolling_window_size": 6,
}
ALGORITHMS = ["lstm", "cnn1d", "random_forest", "xgboost", "light_xgboost"]
LEVELS = ["simple", "medium", "high"]
HORIZON = 1
N_TRIALS_PER_RUN = 20
LAGS_RANGE = (1, 20)
OUTPUT_CSV = "BestParams_15Runs.csv"
NO_PERSIST_FLAGS = {
    "inference_interval_sec": 0,
    "save_artifacts": False,
    "disable_artifact_persistence": True,
    "disable_metrics_persist": True,
    "skip_metrics_persistence": True,
}
def _deep_merge(a: dict, b: dict) -> dict:
    out = dict(a)
    for k, v in (b or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge(out[k], v)
        elif v is not None:
            out[k] = v
    return out
def _build_cfg(algo: str, level: str, lags: int, horizon: int) -> dict:
    cfg = build_training_config(algo, level, lags=lags, horizon=horizon)
    cfg = _deep_merge(cfg, FIXED_FEATURES)
    cfg = _deep_merge(cfg, NO_PERSIST_FLAGS)
    return cfg
def _update_model_params_with_policy(cfg: dict, extra: dict, nested_key: str, lock_keys=set()):
    block = dict(cfg.get(nested_key, {}))
    for k, v in (extra or {}).items():
        if k in lock_keys:
            continue
        block[k] = v
    cfg[nested_key] = block
    for k, v in (extra or {}).items():
        if k in lock_keys:
            continue
        cfg[k] = v
    return cfg


In [5]:

def _suggest_additional_params(trial, algo: str, cfg: dict) -> dict:
    algo = algo.lower()
    lags = trial.suggest_int("lags", LAGS_RANGE[0], LAGS_RANGE[1], step=1)
    extra_top = {}
    lock_structural = set()
    if algo == "lstm":
        lock_structural |= {"num_layers", "initial_units"}
        extra = {
            "dropout": trial.suggest_float("dropout", 0.0, 0.5),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64, 128]),
            "epochs": trial.suggest_int("epochs", 20, 120, step=5),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
            "optimizer": trial.suggest_categorical("optimizer", ["adam", "nadam", "rmsprop"]),
            "loss": trial.suggest_categorical("loss", ["mse", "huber"]),
            "clipnorm": trial.suggest_float("clipnorm", 0.0, 5.0),
            "weight_decay": trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True),
        }
        extra_top["model_params"] = extra
    elif algo == "cnn1d":
        lock_structural |= {"cnn_blocks", "cnn_base_filters", "cnn_kernel_size"}
        extra = {
            "cnn_dropout": trial.suggest_float("cnn_dropout", 0.0, 0.5),
            "cnn_activation": trial.suggest_categorical("cnn_activation", ["relu", "gelu", "tanh"]),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64, 128]),
            "epochs": trial.suggest_int("epochs", 20, 120, step=5),
            "optimizer": trial.suggest_categorical("optimizer", ["adam", "nadam", "rmsprop"]),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
            "clipnorm": trial.suggest_float("clipnorm", 0.0, 5.0),
            "weight_decay": trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True),
            "loss": trial.suggest_categorical("loss", ["huber", "mse"]),
        }
        extra_top["model_params"] = extra
    elif algo == "random_forest":
        lock_structural |= {"n_estimators", "max_depth"}
        extra = {
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 16),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 16),
            "max_features": trial.suggest_float("max_features", 0.2, 1.0),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 0.02),
            "max_samples": trial.suggest_float("max_samples", 0.6, 1.0),
        }
        extra_top["model_params"] = extra
    elif algo in ("xgboost", "light_xgboost"):
        lock_structural |= {"n_estimators", "max_depth"}
        common_extra = {
            "learning_rate": trial.suggest_float("learning_rate", 5e-3, 5e-2, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 1.0, log=True),
        }
        if algo == "xgboost":
            xgb_extra = {
                **common_extra,
                "gamma": trial.suggest_float("gamma", 0.0, 5.0),
                "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 10.0),
                "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
            }
            extra_top["xgb_params"] = xgb_extra
        else:
            lgbm_extra = {
                **{k: v for k, v in common_extra.items() if k != "min_child_weight"},
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
                "max_bin": trial.suggest_int("max_bin", 64, 512, step=32),
            }
            lock_structural |= {"num_leaves"}
            extra_top["lgbm_params"] = lgbm_extra
    return {"lags": lags, "extra_top": extra_top, "lock_structural": lock_structural}


In [6]:

def _extract_val_metric(ret) -> float:
    if isinstance(ret, (float, int)):
        return float(ret)
    if isinstance(ret, dict):
        for k in ["val_mae", "val_mape", "val_rmse", "val_loss", "valid_mae", "valid_mape", "valid_rmse"]:
            if k in ret and isinstance(ret[k], (float, int)):
                return float(ret[k])
    for k in ["val_mae_", "val_mape_", "val_rmse_", "val_loss_"]:
        if hasattr(ret, k):
            v = getattr(ret, k)
            if isinstance(v, (float, int)):
                return float(v)
    return float("inf")
def _import_trainer(algo: str):
    module, clsname, folder_flag = TRAINER_MAP[algo]
    mod = importlib.import_module(module)
    Trainer = getattr(mod, clsname)
    return Trainer, folder_flag
def train_and_score(algo: str, level: str, lags: int, horizon: int, extra_top: dict, lock_structural=None) -> float:
    cfg = _build_cfg(algo, level, lags=lags, horizon=horizon)
    lock = set(lock_structural or [])
    if "model_params" in extra_top:
        cfg = _update_model_params_with_policy(cfg, extra_top["model_params"], "model_params", lock_keys=lock)
    if "xgb_params" in extra_top:
        cfg = _update_model_params_with_policy(cfg, extra_top["xgb_params"], "xgb_params", lock_keys=lock)
    if "lgbm_params" in extra_top:
        cfg = _update_model_params_with_policy(cfg, extra_top["lgbm_params"], "lgbm_params", lock_keys=lock)
    Trainer, folder_flag = _import_trainer(algo)
    trainer = Trainer(config=cfg, folder_flag=folder_flag)
    try:
        ret = trainer.run(save_artifacts=False, return_metrics=True)
    except TypeError:
        ret = trainer.run(save_artifacts=False)
    except Exception as e:
        warnings.warn(f"Training failed for {algo}/{level} (lags={lags}): {e}")
        return float("inf")
    return _extract_val_metric(ret)


## Vorab-Check: Dataset & Spalten (Pipeline-Style)

In [7]:

# Optional: manueller Override-Pfad (leer lassen, wenn nicht verwendet)
OVERRIDE_DATASET_PATH = ""  # z.B.: r"C:\...\Input\Input_Data\mqtt_data_filtered.csv"

from pathlib import Path

def _resolve_dataset_path_pipeline_style(dataset_name: str, algo_hint: str = "lstm", level_hint: str = "simple") -> Path | None:
    """ Bestimme den Dataset-Pfad so, wie es die Pipeline macht. """
    # 0) Manueller Override
    if OVERRIDE_DATASET_PATH:
        p = Path(OVERRIDE_DATASET_PATH)
        if p.exists():
            return p.resolve()
        else:
            print("⚠️ OVERRIDE_DATASET_PATH gesetzt, aber Datei nicht gefunden:", p)

    # 1) Config zusammenbauen
    try:
        cfg = _build_cfg(algo_hint, level_hint, lags=LAGS_RANGE[0], horizon=HORIZON)
    except Exception:
        cfg = {"paths": CONFIG_PATH.get("paths", {})}

    dataset = str(dataset_name or "").strip()
    if not dataset:
        return None

    candidates = []
    paths = (cfg.get("paths") or {}) if isinstance(cfg, dict) else {}

    # 2) Direkte Keys aus CONFIG_PATH['paths'] inkl. 'input_data' und 'base'
    for key in ["input_data", "input", "data", "Datasets", "dataset", "raw", "data_dir", "datasets_dir", "base"]:
        p = paths.get(key)
        if p:
            candidates.append(Path(p) / dataset)

    # 3) Häufige Unterordner relativ zu 'base'
    base = Path(paths.get("base")) if paths.get("base") else None
    if base and base.exists():
        candidates += [
            base / "Input" / "Input_Data" / dataset,
            base / "Input" / dataset,
            base / "Datasets" / dataset,
            base / "data" / dataset,
        ]

    # 4) Generische Orte im Projekt
    candidates += [
        Path(dataset),
        Path("data") / dataset,
        Path("Datasets") / dataset,
        Path("/mnt/data") / dataset,
    ]

    for c in candidates:
        try:
            if c.exists():
                return c.resolve()
        except Exception:
            continue
    return None

REQ_COLS = set(FIXED_FEATURES["base_features"] + [FIXED_FEATURES["target_feature"]] + FIXED_FEATURES["time_features"])
ds = _resolve_dataset_path_pipeline_style(BASE_COMMON.get("dataset", "mqtt_data_filtered.csv"), algo_hint=ALGORITHMS[0], level_hint=LEVELS[0])
if ds is None:
    print(f"⚠️ Dataset '{BASE_COMMON.get('dataset')}' nicht über CONFIG_PATH['paths']/Projektstruktur gefunden. "
          f"Bitte Datei nach 'Input/Input_Data' legen oder OVERRIDE_DATASET_PATH setzen.")
else:
    print("Gefundenes Dataset (Pipeline-Style):", ds)
    try:
        import pandas as pd
        probe = pd.read_csv(ds, nrows=5)
        missing = [c for c in REQ_COLS if c not in probe.columns]
        print("Verfügbare Spalten (Ausschnitt):", list(probe.columns)[:10], "...")
        if missing:
            print("❌ Fehlende Pflichtspalten:", missing)
        else:
            print("✅ Alle Pflichtspalten vorhanden.")
    except Exception as e:
        print("Hinweis: Konnte Spalten nicht prüfen:", e)


Gefundenes Dataset (Pipeline-Style): C:\DEV\RevPi_ML\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Verfügbare Spalten (Ausschnitt): ['time', 'datetime', 'Group4-2_S6_MassFlowRate', 'Group4-2_S6_FlowVelocity', 'Group4-2_S6_Volume', 'Group4-2_S6_VolumetricFlowRate', 'Group4-2_S6_Mass', 'Group4-2_S6_Energy', 'Group4-2_S6_Temperature', 'Group4-2_S6_Pressure'] ...
✅ Alle Pflichtspalten vorhanden.


## Optuna-Optimierung (15 Läufe, Random Search)

In [8]:

best_rows = []
def make_objective(algo: str, level: str, horizon: int):
    dummy_cfg = _build_cfg(algo, level, lags=LAGS_RANGE[0], horizon=horizon)
    def _objective(trial: optuna.trial.Trial) -> float:
        sug = _suggest_additional_params(trial, algo, dummy_cfg)
        lags = sug['lags']
        extra_top = sug['extra_top']
        lock_structural = sug.get('lock_structural', set())
        return train_and_score(algo, level, lags, horizon, extra_top, lock_structural)
    return _objective
for algo in ALGORITHMS:
    for level in LEVELS:
        print(f"\n=== Study: {algo} / {level} ===")
        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.RandomSampler(seed=42))
        study.optimize(make_objective(algo, level, HORIZON), n_trials=N_TRIALS_PER_RUN, show_progress_bar=False)
        best = study.best_trial
        row = {"algorithm": algo, "complexity": level, "horizon": HORIZON, "best_value": best.value}
        for k, v in best.params.items():
            row[k] = v
        best_rows.append(row)
df_best = pd.DataFrame(best_rows)
df_best.to_csv(OUTPUT_CSV, index=False)
df_best


[I 2025-08-25 00:47:55,291] A new study created in memory with name: no-name-46ec35fb-d187-4994-9fe8-220cd4239f63
[W 2025-08-25 00:47:55,315] Trial 0 failed with parameters: {'lags': 8, 'dropout': 0.4753571532049581, 'batch_size': 16, 'epochs': 25, 'learning_rate': 0.0029621516588303515, 'optimizer': 'nadam', 'loss': 'mse', 'clipnorm': 1.0616955533913808, 'weight_decay': 8.111941985431907e-08} because of the following error: ImportError("cannot import name 'pipeline_utils' from 'ML_Helpfunctions' (C:\\DEV\\RevPi_ML\\ML_Edge_Device\\ML_Helpfunctions\\__init__.py)").
Traceback (most recent call last):
  File "c:\DEV\RevPi_ML\ML_Edge_Device\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\WanDa\AppData\Local\Temp\ipykernel_46032\3235722641.py", line 9, in _objective
    return train_and_score(algo, level, lags, horizon, extra_top, lock_structural)
           ^^^^^^^^^^^^^^^^^^^^


=== Study: lstm / simple ===


ImportError: cannot import name 'pipeline_utils' from 'ML_Helpfunctions' (C:\DEV\RevPi_ML\ML_Edge_Device\ML_Helpfunctions\__init__.py)


### Hinweise
- **Persistenz**: Modelle, Scaler und Fehlermetriken werden **nicht** gespeichert. Es wird nur `BestParams_15Runs.csv` geschrieben.
- **Feature-Flags** (`include_roll_mean/std`, `rolling_window_size`) sind **fix** gesetzt und **nicht** Teil der Optimierung.
- **Komplexitäts-Parameter** aus den Presets werden **nicht** verändert; HPO überschreibt nur nicht-strukturelle Trainings-/Reg-Parameter.
- Dataset-Pfade werden **wie in der Pipeline** über `CONFIG_PATH['paths']` abgeleitet; der Vorab-Check ist informativ.
